# Out-of-bag error and variable importance for random forests

A random forest grows each tree on a bootstrap sample, so about a third of the cases are left out of each tree. Breiman (2001) used these out-of-bag cases twice: to estimate the forest's error without a test set, and to measure how much the forest relies on each input, by permuting that input among them. DecisionTree.jl grows forests, but has neither. Both are written out here as documented functions and tested against Breiman's published errors: Table 2's Selection, Single input and One tree columns on 19 data sets, which rest on the out-of-bag errors, and the test errors of forests on the inputs his importance picked out (Section 10).

The trees are grown with DecisionTree.jl. The tests use 14 real data sets from PollisDatasets, four simulated ones generated as Breiman did, and the zip-code digits, downloaded only if you agree.

In [ ]:
Pollis.packages("DecisionTree", "StatsBase", "PollisDatasets", "Plots", "CodecZlib", "ProgressLogging")

## Set-up

Breiman's errors are averages of 100 random runs, so `compare` prints each Julia average with its standard error, rounded beside the published value, and whether the two differ by less than two standard errors of a difference of two such averages (√2 se), Breiman's being one too. `repetitions` sets the number of runs: 100 as in the paper. Every run has its own seed, so the results do not depend on the number of threads; to use more, set **julia.NumThreads** to `auto` in the settings and restart the kernel.

`inputs` reads a data set as Breiman used it: the inputs as a matrix, a yes vote as 1, the classes numbered, and the few missing values replaced by the median of their input.

In [ ]:
using DecisionTree, StatsBase, PollisDatasets, Plots, CodecZlib, ProgressLogging, DelimitedFiles, Downloads, Printf, Random, Statistics
using DecisionTree: Leaf, Node

"""
	compare(title, rows)

Print each Julia error rate, in percent, with its standard error, rounded beside the published value, and whether they differ by less than
two standard errors of a difference (√2 se). Each row is `(label, value, se, published)`.
"""
function compare(title, rows)
	@printf("\n%s\n%-16s%10s%8s%10s%11s\n", title, "", "Julia", "se", "rounded", "published")
	for (label, value, se, published) in rows
		z = (value - parse(Float64, published)) / (sqrt(2) * se)
		@printf("%-16s%10.3f%8.3f%10.1f%11s  %s\n", label, value, se, value, published, abs(z) < 2 ? "within noise" : @sprintf("DIFFERENT (%.1f se)", z))
	end
end

repetitions = 100  # runs per data set, as in the paper
println("threads: ", Threads.nthreads())

"""Run `f(rng)` for runs 1:n, each with its own seed, on all threads, with a progress bar named `name`."""
function runs(f, n; name="runs")
	r = Vector{Any}(undef, n)
	finished = Threads.Atomic{Int}(0)
	@withprogress name=name begin
		Threads.@threads for i in 1:n
			r[i] = f(Xoshiro(i))
			@logprogress (Threads.atomic_add!(finished, 1) + 1) / n
		end
	end
	r
end

const CLASS = Dict("glass" => :type, "breastwisconsin" => :class, "pima" => :diabetes, "sonar" => :class, "vowel" => :vowel,
	"ionosphere" => :class, "vehicle" => :class, "soybean" => :disease, "germancredit" => :credit, "image" => :class,
	"ecoli" => :site, "votes" => :party, "liver" => :selector, "letters" => :letter, "satimage" => :class)
const NOTINPUT = [:id, :protein, :speaker, :segment, :set]  # identifiers and the train/test marker

"""The inputs of data set `id` as a matrix, missing values replaced by the median of their input, and the classes numbered 1, 2, …"""
function inputs(id)
	d = dataset(id)
	cols = [k for k in keys(d) if k ∉ NOTINPUT && k != CLASS[id]]
	X = reduce(hcat, [begin
		c = [v isa AbstractString ? Float64(v == "y") : v for v in d[k]]  # votes: yes = 1, no = 0
		m = median(skipmissing(c))
		Float64[ismissing(v) ? m : v for v in c]
	end for k in cols])
	labels = string.(d[CLASS[id]])
	names = sort(unique(labels))
	X, indexin(labels, names), length(names)
end

## The method

### Forest-RI and its out-of-bag error

Forest-RI (Breiman 2001, p. 9) grows each tree on a bootstrap sample of the n training cases. At each node it draws F inputs at random and splits on the best of them by the Gini criterion, and it grows each tree to the end, unpruned. The forest classifies by majority vote.

Run down the trees that did not see them, the out-of-bag cases give the **out-of-bag error**: the error of the forest's vote on each case, counting only those trees, an estimate of the test error that needs no test set (Section 3.1, p. 11). The same cases give the error of each tree on its own, Table 2's "One tree". `forest` returns the trees and both errors. `classify` runs cases down a tree without recursion, since the deepest trees overflow the stack of DecisionTree.jl's recursive `apply_tree`.

In [ ]:
"""Run each row of `X` down tree `t`."""
function classify(t, X::AbstractMatrix)
	t isa DecisionTree.Root && (t = t.node)
	map(eachrow(X)) do x
		s = t
		while s isa Node
			s = x[s.featid] < s.featval ? s.left : s.right
		end
		s.majority
	end
end

"""The class with the most votes in each row of `votes`."""
winner(votes) = [argmax(r) for r in eachrow(votes)]

"""A Forest-RI tree on the cases `boot`: unpruned, Gini, each node split on the best of `F` random inputs."""
grow(X, y, boot, F, rng) = build_tree(y[boot], X[boot, :], F, -1, 1, 2, 0.0; loss=DecisionTree.util.gini, rng=rng, impurity_importance=false)

"""
	forest(X, y, C, F, ntree, rng)

Forest-RI with `ntree` trees and `F` random inputs per node, for `C` classes.
Returns the trees, the out-of-bag error of the forest, and the average out-of-bag error of its trees.
"""
function forest(X, y, C, F, ntree, rng)
	n = length(y)
	votes = zeros(n, C)
	trees = Vector{Any}(undef, ntree)
	single = 0.0
	for b in 1:ntree
		boot = rand(rng, 1:n, n)
		trees[b] = grow(X, y, boot, F, rng)
		oob = setdiff(1:n, boot)
		p = classify(trees[b], X[oob, :])
		single += mean(p .!= y[oob])
		for (i, c) in zip(oob, p)
			votes[i, c] += 1
		end
	end
	seen = vec(sum(votes, dims=2)) .> 0
	trees, mean(winner(votes[seen, :]) .!= y[seen]), single / ntree
end

"""The majority vote of `trees` on each row of `X`, with `C` classes."""
function vote(trees, X, C)
	votes = zeros(size(X, 1), C)
	for t in trees
		for (i, c) in enumerate(classify(t, X))
			votes[i, c] += 1
		end
	end
	winner(votes)
end

### Permutation importance

To see which inputs a forest relies on, Breiman (Section 10, pp. 19-21) permutes the values of input m among the out-of-bag cases of each tree, after growing it, and runs them down the tree again. The out-of-bag vote with input m so noised up is compared with the true classes, and the percent increase over the out-of-bag error with all inputs intact measures the importance of m.

In [ ]:
"""
	importance(X, y, C, F, ntree, rng)

The percent increase in the out-of-bag error of a Forest-RI when each input in turn is permuted among the out-of-bag cases of every tree.
"""
function importance(X, y, C, F, ntree, rng)
	n, M = size(X)
	votes, noised = zeros(n, C), zeros(n, C, M)
	for b in 1:ntree
		boot = rand(rng, 1:n, n)
		t = grow(X, y, boot, F, rng)
		oob = setdiff(1:n, boot)
		for (i, c) in zip(oob, classify(t, X[oob, :]))
			votes[i, c] += 1
		end
		for m in 1:M
			Xm = X[oob, :]
			Xm[:, m] = shuffle(rng, Xm[:, m])
			for (i, c) in zip(oob, classify(t, Xm))
				noised[i, c, m] += 1
			end
		end
	end
	seen = vec(sum(votes, dims=2)) .> 0
	base = mean(winner(votes[seen, :]) .!= y[seen])
	[100 * (mean(winner(noised[seen, :, m]) .!= y[seen]) - base) / base for m in 1:M]
end

## Tests

### Table 2

On each data set Breiman grew two forests of 100 trees, one with F = 1 and one with F = int(log₂ M + 1), M the number of inputs, and took the test error of the one with the lower **out-of-bag** error: the **Selection** column. **Single input** is the test error of the forest with F = 1, and **One tree** the out-of-bag error of the single trees of the selected forest (pp. 10-11). Selection and One tree thus test the out-of-bag errors; Single input tests the forest they are computed on.

The small data sets set aside a random 10% as the test set, 100 times. Letters, sat-images and zip-code have their own test sets and are one run each, with 200 trees for zip-code; their standard errors are the binomial ones, sqrt(e(100 − e)/n). The zip-code digits, 9,298 handwritten digits of 16 × 16 grey levels, are not in PollisDatasets: set `download_zipcode = true` to download them, 2.3 MB, from the site of Hastie, Tibshirani and Friedman's *The Elements of Statistical Learning* into ~/.pollis/data/zipcode; otherwise the zip-code row is left out.

In [ ]:
"""Selection, Single input and One tree, in percent, for forests of `ntree` trees trained on `train` and tested on `test`."""
function table2(X, y, C, train, test, rng; ntree=100)
	F = floor(Int, log2(size(X, 2)) + 1)
	t1, oob1, one1 = forest(X[train, :], y[train], C, 1, ntree, rng)
	tF, oobF, oneF = forest(X[train, :], y[train], C, F, ntree, rng)
	single = mean(vote(t1, X[test, :], C) .!= y[test])
	selection = oobF < oob1 ? mean(vote(tF, X[test, :], C) .!= y[test]) : single
	100 .* [selection, single, oobF < oob1 ? oneF : one1]
end

published = Dict("glass" => ("20.6", "21.2", "36.9"), "breastwisconsin" => ("2.9", "2.7", "6.3"), "pima" => ("24.2", "24.3", "33.1"),
	"sonar" => ("15.9", "18.0", "31.7"), "vowel" => ("3.4", "3.3", "30.4"), "ionosphere" => ("7.1", "7.5", "12.7"),
	"vehicle" => ("25.8", "26.4", "33.1"), "germancredit" => ("24.4", "26.2", "33.3"), "image" => ("2.1", "2.7", "6.4"),
	"ecoli" => ("12.8", "13.0", "24.5"), "votes" => ("4.1", "4.6", "7.4"), "liver" => ("25.1", "24.7", "40.6"),
	"letters" => ("3.5", "4.7", "19.8"), "satimage" => ("8.6", "10.5", "17.2"), "zip" => ("6.3", "7.8", "20.6"),
	"waveform" => ("17.2", "17.3", "34.0"), "twonorm" => ("3.9", "3.9", "24.7"), "threenorm" => ("17.5", "17.5", "38.4"),
	"ringnorm" => ("4.9", "4.9", "25.7"))
columns = ("Selection", "Single input", "One tree")
results = Dict{String, Any}()  # data set => (averages, standard errors)

function report(id, title, m, se)
	results[id] = (m, se)
	compare("$id (Table 2, p. 11), $title", [(columns[j], m[j], se[j], published[id][j]) for j in 1:3])
end

small = ["glass", "breastwisconsin", "pima", "sonar", "vowel", "ionosphere", "vehicle", "germancredit", "image", "ecoli", "votes", "liver"]
for id in small
	X, y, C = inputs(id)
	R = reduce(hcat, runs(repetitions; name=id) do rng
		n = length(y)
		test = randperm(rng, n)[1:round(Int, n / 10)]
		table2(X, y, C, setdiff(1:n, test), test, rng)
	end)'
	report(id, "$(size(R, 1)) runs", vec(mean(R, dims=1)), vec(std(R, dims=1)) ./ sqrt(size(R, 1)))
end

In [ ]:
download_zipcode = false  # set to true to download the zip-code data (2.3 MB, once)
zipdir = joinpath(homedir(), ".pollis", "data", "zipcode")
if download_zipcode
	mkpath(zipdir)
	for f in ("zip.train.gz", "zip.test.gz")
		isfile(joinpath(zipdir, f)) || Downloads.download("https://hastie.su.domains/ElemStatLearn/datasets/$f", joinpath(zipdir, f))
	end
end

"""A zip-code file: the digit, then the 256 grey levels, one case per line."""
function zipread(f)
	d = readdlm(GzipDecompressorStream(open(f)), Float64)[:, 1:257]
	d[:, 2:end], Int.(d[:, 1]) .+ 1
end

"""One run on a data set with its own test set; the standard errors are binomial."""
function largerun(id, X, y, C, train, test; ntree=100)
	e = @withprogress name=id table2(X, y, C, train, test, Xoshiro(2001); ntree=ntree)
	report(id, "$(length(train)) training and $(length(test)) test cases", e, sqrt.(e .* (100 .- e) ./ length(test)))
end

X, y, C = inputs("letters")
largerun("letters", X, y, C, 1:15000, 15001:20000)  # the first 15,000 cases train: the paper does not say which
X, y, C = inputs("satimage")
set = dataset("satimage").set
largerun("satimage", X, y, C, findall(==("train"), set), findall(==("test"), set))
if isfile(joinpath(zipdir, "zip.test.gz"))
	X1, y1 = zipread(joinpath(zipdir, "zip.train.gz"))
	X2, y2 = zipread(joinpath(zipdir, "zip.test.gz"))
	largerun("zip", vcat(X1, X2), vcat(y1, y2), 10, 1:length(y1), length(y1)+1:length(y1)+length(y2); ntree=200)
end

The four simulated data sets are those of the arcing paper (Breiman 1998, p. 7), with two classes, or three for waveform, equally likely:

- **twonorm**: 20 inputs, normal with identity covariance and mean (a, …, a) for class 1, (−a, …, −a) for class 2, a = 2/√20.
- **threenorm**: class 1 an equal mixture of the normals with means (a, …, a) and (−a, …, −a), class 2 the normal with mean (a, −a, a, −a, …), a = 2/√20.
- **ringnorm**: class 1 normal with mean 0 and covariance 4 times the identity, class 2 normal with mean (a, …, a) and the identity, a = 1/√20.
- **waveform** (Breiman et al. 1984, p. 49): 21 inputs, each class a random mixture u h + (1 − u) h′ of two of three triangular waves plus N(0, 1) noise: class 1 mixes h₁ and h₂, class 2 h₁ and h₃, class 3 h₂ and h₃, with u uniform on (0, 1), h₁(i) = max(6 − |i − 11|, 0), h₂(i) = h₁(i − 4) and h₃(i) = h₁(i + 4).

Each of 50 runs draws a training set of 300 cases and a test set of 3,000 (p. 11).

In [ ]:
function twonorm(n, rng)
	a = 2 / sqrt(20)
	y = rand(rng, 1:2, n)
	randn(rng, n, 20) .+ a .* ifelse.(y .== 1, 1, -1), y
end

function threenorm(n, rng)
	a = 2 / sqrt(20)
	y = rand(rng, 1:2, n)
	alternating = [isodd(j) ? a : -a for j in 1:20]
	X = randn(rng, n, 20)
	for i in 1:n
		X[i, :] .+= y[i] == 2 ? alternating : fill(rand(rng, Bool) ? a : -a, 20)
	end
	X, y
end

function ringnorm(n, rng)
	a = 1 / sqrt(20)
	y = rand(rng, 1:2, n)
	X = randn(rng, n, 20)
	ifelse.(y .== 1, 2 .* X, X .+ a), y
end

function waveform(n, rng)
	h1 = [max(6 - abs(i - 11), 0) for i in 1:21]
	h2 = [max(6 - abs(i - 4 - 11), 0) for i in 1:21]
	h3 = [max(6 - abs(i + 4 - 11), 0) for i in 1:21]
	mixtures = ((h1, h2), (h1, h3), (h2, h3))
	y = rand(rng, 1:3, n)
	X = randn(rng, n, 21)
	for i in 1:n
		u = rand(rng)
		h, h′ = mixtures[y[i]]
		X[i, :] .+= u .* h .+ (1 - u) .* h′
	end
	X, y
end

for (id, simulate, C) in (("waveform", waveform, 3), ("twonorm", twonorm, 2), ("threenorm", threenorm, 2), ("ringnorm", ringnorm, 2))
	R = reduce(hcat, runs(50; name=id) do rng
		X, y = simulate(300, rng)
		Xt, yt = simulate(3000, rng)
		table2([X; Xt], [y; yt], C, 1:300, 301:3300, rng)
	end)'
	report(id, "$(size(R, 1)) runs", vec(mean(R, dims=1)), vec(std(R, dims=1)) ./ sqrt(size(R, 1)))
end

### Section 10: the inputs that matter

On diabetes, with F = 1 and one forest of 1,000 trees, Breiman's importance singles out input 2 (plasma glucose), followed by 8 (age) and 6 (body mass) (Figure 4, p. 20). The figure has no printed values, but the test errors of forests on the chosen inputs are printed (p. 20), from the 100 runs of Table 2: 29.7% with input 2 alone, 23.1% with all eight, 29.4% with inputs 2 and 8, and 26.4% with inputs 2 and 6. On the 1984 House votes, with F = 5, input 4 (freezing physicians' fees) stands out (Figure 6, p. 21), and a forest on it alone has a test error of 4.3%.

In [ ]:
"""The Selection error of Table 2 for forests on the inputs `vars` only, over `repetitions` runs of a 10% test set."""
function selectionerror(X, y, C, vars)
	r = runs(repetitions; name="inputs $(join(vars, ", "))") do rng
		n = length(y)
		test = randperm(rng, n)[1:round(Int, n / 10)]
		table2(X[:, vars], y, C, setdiff(1:n, test), test, rng)[1]
	end
	mean(r), std(r) / sqrt(length(r))
end

X, y, C = inputs("pima")
diabetes = importance(X, y, C, 1, 1000, Xoshiro(10))
println("diabetes, the inputs by importance: ", sortperm(diabetes, rev=true))
compare("diabetes, forests on chosen inputs (p. 20)", [(label, selectionerror(X, y, C, vars)..., p) for (label, vars, p) in
	(("input 2", [2], "29.7"), ("all inputs", 1:8, "23.1"), ("inputs 2 and 8", [2, 8], "29.4"), ("inputs 2 and 6", [2, 6], "26.4"))])

X, y, C = inputs("votes")
votes = importance(X, y, C, 5, 1000, Xoshiro(10))
println("\nvotes, the inputs by importance: ", sortperm(votes, rev=true))
compare("votes, a forest on input 4 alone (p. 21)", [("input 4", selectionerror(X, y, C, [4])..., "4.3")])
plot(bar(1:8, diabetes, xticks=1:8, title="diabetes (Figure 4, p. 20)"), bar(1:16, votes, xticks=1:16, title="votes (Figure 6, p. 21)"),
	xlabel="input", ylabel="% increase in error", label="", size=(760, 300))

## Example

The out-of-bag error is meant to track the test error without a test set. On one split of the image segmentation data, both as the forest grows: they settle at the same level.

In [ ]:
X, y, C = inputs("image")
rng = Xoshiro(3)
test = randperm(rng, length(y))[1:500]
train = setdiff(1:length(y), test)
sizes = [5, 10, 20, 40, 80, 160]
errors = [begin
	trees, oob, _ = forest(X[train, :], y[train], C, 5, k, Xoshiro(k))
	100 .* (oob, mean(vote(trees, X[test, :], C) .!= y[test]))
end for k in sizes]
plot(sizes, first.(errors), label="out-of-bag", xscale=:log10, xticks=(sizes, string.(sizes)), xlabel="trees", ylabel="error %", marker=:circle, size=(560, 330))
plot!(sizes, last.(errors), label="test set", marker=:circle)

## What differs

**Selection and Single input.** In one run of this notebook, 34 of the 38 values were within noise of Table 2. The four that were not:

- liver Selection: 27.3 vs 25.1 (2.0 se);
- twonorm Single input: 3.7 vs 3.9 (2.1 se);
- ringnorm Selection and Single input: 4.4 vs 4.9 and 4.3 vs 4.9 (2.4 and 4.7 se).

Both ringnorm values are lower than Breiman's.

**One tree.** In the same run, 12 of the 19 values differed by more than two standard errors, most of them by about one percentage point (for example pima 31.9 vs 33.1, liver 39.3 vs 40.6). Vowel is the exception, at 36.1 vs 30.4. The test is strict here: each value averages 100 trees over 100 runs, so the standard errors are small, and small differences between DecisionTree.jl and Breiman's tree code (tie-breaking, split points) show up as several standard errors.

**Section 10.** Diabetes inputs 2 / all / 2 and 8 gave 29.9 / 24.0 / 30.2 vs 29.7 / 23.1 / 29.4, within noise. Inputs 2 and 6 gave 28.1 vs 26.4 (2.4 se). Votes input 4 gave 4.7 vs 4.3, within noise. The importance ranking put input 2 first for diabetes and input 4 first for votes, as Breiman found.

## References

- Breiman, L. (1998). Arcing classifiers. *The Annals of Statistics* 26(3), 801-849. doi:10.1214/aos/1024691079
- Breiman, L. (2001). Random forests. *Machine Learning* 45(1), 5-32. doi:10.1023/A:1010933404324
- Breiman, L., Friedman, J. H., Olshen, R. A. and Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.
- Hastie, T., Tibshirani, R. and Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed. Springer: the zip-code data.